In [ ]:
pip install transformers datasets torch scikit-learn

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:
dataset = load_dataset("csv", data_files="/content/amazon_polarity.csv")
dataset = dataset["train"]

print(dataset.column_names)

In [ ]:
if "reviews.text" in dataset.column_names:
    dataset = dataset.rename_column("reviews.text", "text")

if "reviews.rating" in dataset.column_names:
    dataset = dataset.rename_column("reviews.rating", "label")

In [ ]:
dataset = dataset.select_columns(["text", "label"])

In [ ]:
dataset = dataset.filter(lambda x: x["text"] is not None)
dataset = dataset.filter(lambda x: x["label"] is not None)

In [ ]:
from datasets import Value

def convert_label(example):
    example["label"] = int(1 if int(example["label"]) >= 4 else 0)
    return example

dataset = dataset.map(convert_label)


dataset = dataset.cast_column("label", Value("int64"))

In [ ]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

print(train_dataset[0]["label"], type(train_dataset[0]["label"]))


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# Important fix
model.config.problem_type = "single_label_classification"

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    return "Positive" if torch.argmax(probs) == 1 else "Negative"

print(predict("This product is amazing!"))